In [1]:
import sys, time, csv
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.models.embeddings import encode_news

RAW = ROOT / "data" / "raw" / "MINDlarge_test"
OUT = ROOT / "data" / "processed" / "large"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
NEWS_COLS = ["news_id", "category", "subcategory", "title", "abstract",
             "url", "title_entities", "abstract_entities"]
BEH_COLS = ["impression_id", "user_id", "time", "history", "impressions"]

for f in ("news.tsv", "behaviors.tsv"):
    p = RAW / f
    print(f"{f:15s} {'OK' if p.exists() else 'NO ENCONTRADO'}"
          + (f"  {p.stat().st_size / 1e6:8.1f} MB" if p.exists() else ""))

muestra = pd.read_csv(RAW / "behaviors.tsv", sep="\t", header=None, names=BEH_COLS,
                      quoting=csv.QUOTE_NONE, nrows=3)
print(muestra[["impression_id", "time", "impressions"]].to_string())

with open(RAW / "behaviors.tsv", encoding="utf-8") as fh:
    n_imp = sum(1 for _ in fh)
print(f"\nImpresiones en test: {n_imp:,}")

news.tsv        OK     101.3 MB
behaviors.tsv   OK    1460.6 MB
   impression_id                    time                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             impressions
0              1  11/19/2019 11:37:45 AM                                                                                                                                                                                                                                                                               

In [2]:
test_news = (pd.read_csv(RAW / "news.tsv", sep="\t", header=None, names=NEWS_COLS,
                         quoting=csv.QUOTE_NONE, usecols=["news_id", "title", "abstract"])
               .drop_duplicates("news_id"))
prev = pd.read_parquet(OUT / "news_emb_minilm.parquet")

nuevas = test_news[~test_news["news_id"].isin(prev["news_id"])].reset_index(drop=True)
nuevas["text"] = (nuevas["title"].fillna("") + " " + nuevas["abstract"].fillna("")).str.strip()
print(f"Noticias en test: {len(test_news):,} | ya codificadas: {len(test_news) - len(nuevas):,} "
      f"| nuevas: {len(nuevas):,}")

t0 = time.time()
emb_nuevas = encode_news(nuevas, MODEL_NAME)
print(f"Embeddings nuevos: {emb_nuevas.shape} en {(time.time() - t0) / 60:.1f} min")

Noticias en test: 120,961 | ya codificadas: 94,733 | nuevas: 26,228


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/205 [00:00<?, ?it/s]

Embeddings nuevos: (26228, 384) en 11.8 min


In [3]:
todo = pd.concat([
    prev,
    pd.DataFrame({"news_id": nuevas["news_id"], "emb": list(emb_nuevas.astype(np.float32))}),
], ignore_index=True)

destino = OUT / "news_emb_minilm_all.parquet"
todo.to_parquet(destino, index=False)
print(f"Catálogo completo: {len(todo):,} noticias → {destino} ({destino.stat().st_size / 1e6:.0f} MB)")

faltan = set(test_news["news_id"]) - set(todo["news_id"])
print("Noticias del test sin embedding:", len(faltan))     # debe ser 0

Catálogo completo: 130,379 noticias → c:\DevProjects\MMDS\RecSysMIND\data\processed\large\news_emb_minilm_all.parquet (202 MB)
Noticias del test sin embedding: 0
